In [0]:
from pyspark.sql import types as t
from pyspark.sql import functions as f

Usando o Struct filed e struct type

In [0]:
# id,created_at,first_name,last_name,email,cell_phone,country,state,street,number,additionals

schemas = t.StructType([
    t.StructField('id', t.IntegerType(),nullable=False),
    t.StructField('created_at', t.DateType(), nullable=True),
    t.StructField('first_name', t.StringType(), nullable=True),
    t.StructField('last_name',  t.StringType(), nullable=True),
    t.StructField('email',      t.StringType(), nullable=True),
    t.StructField('cell_phone', t.StringType(), nullable=True),
    t.StructField('country',    t.StringType(), nullable=True),
    t.StructField('state',      t.StringType(), nullable=True),
    t.StructField('street',     t.StringType(), nullable=True),
    t.StructField('number',     t.StringType(), nullable=True),
    t.StructField('additionals', t.StringType(), nullable=True)
])

In [0]:
display(dbutils.fs.ls("/Volumes/learn_databricks/schema/volume/Clientes"))

path,name,size,modificationTime
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/Clientes.csv,Clientes.csv,12356,1783608175000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/json/,json/,0,1783637261026
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/parquet/,parquet/,0,1783637261026


In [0]:
path_csv = "/Volumes/learn_databricks/schema/volume/Clientes/Clientes.csv"

In [0]:
# Lendo arquivo .csv

data = spark\
        .read\
        .format("csv")\
        .options(
            header=True,
            inferSchema=False,
            delimiter=",",
            skipLeadingRows=1
        )\
        .schema(schema=schemas)\
        .load(path_csv)

In [0]:
data.printSchema()

root
 |-- id: integer (nullable = true)
 |-- created_at: date (nullable = true)
 |-- first_name: string (nullable = true)
 |-- last_name: string (nullable = true)
 |-- email: string (nullable = true)
 |-- cell_phone: string (nullable = true)
 |-- country: string (nullable = true)
 |-- state: string (nullable = true)
 |-- street: string (nullable = true)
 |-- number: string (nullable = true)
 |-- additionals: string (nullable = true)



In [0]:
data.select("*")

DataFrame[id: int, created_at: date, first_name: string, last_name: string, email: string, cell_phone: string, country: string, state: string, street: string, number: string, additionals: string]

In [0]:
data.select("state").distinct().show()

+------------------+
|             state|
+------------------+
|           Sergipe|
|      Minas Gerais|
|        Pernambuco|
|             Amapá|
|         São Paulo|
|             Ceará|
|          Rondônia|
|    Espírito Santo|
|              Acre|
| Rio Grande do Sul|
|    Santa Catarina|
|  Distrito Federal|
|           Alagoas|
|            Paraná|
|             Bahia|
|              NULL|
|          Amazonas|
|Mato Grosso do Sul|
|             Goiás|
|             Piauí|
+------------------+
only showing top 20 rows


In [0]:
data.repartition(7, "state").write.mode("overwrite").format("csv").save("/Volumes/learn_databricks/schema/volume/Clientes/partition_country")

In [0]:
display(dbutils.fs.ls("/Volumes/learn_databricks/schema/volume/Clientes/partition_country"))

path,name,size,modificationTime
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_SUCCESS,_SUCCESS,0,1783637753000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_committed_1859925246994949121,_committed_1859925246994949121,557,1783637694000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_committed_6165914234789864239,_committed_6165914234789864239,390,1783637526000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_committed_8117288741993947996,_committed_8117288741993947996,824,1783637753000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_committed_8327951429126211191,_committed_8327951429126211191,113,1783637415000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_started_1859925246994949121,_started_1859925246994949121,0,1783637694000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_started_6165914234789864239,_started_6165914234789864239,0,1783637525000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_started_8117288741993947996,_started_8117288741993947996,0,1783637752000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/_started_8327951429126211191,_started_8327951429126211191,0,1783637415000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country/part-00000-tid-8117288741993947996-3e356283-fb49-4917-8a13-98582ce704a4-192-1-c000.csv,part-00000-tid-8117288741993947996-3e356283-fb49-4917-8a13-98582ce704a4-192-1-c000.csv,1516,1783637752000


In [0]:
# Lendo arquivo .csv

data = spark\
        .read\
        .format("csv")\
        .options(
            header=True,
            inferSchema=False,
            delimiter=",",
            skipLeadingRows=1
        )\
        .schema(schema=schemas)\
        .load("/Volumes/learn_databricks/schema/volume/Clientes/partition_country")

In [0]:
data.limit(3).display()

id,created_at,first_name,last_name,email,cell_phone,country,state,street,number,additionals
3,2018-01-17,Daniela,Avelino,daniela@exemplo.com,9 4642-9486,Brasil,Mato Grosso,null,null,null
5,2018-01-05,Marcelo,Barroso,null,9 2830-2088,Brasil,Rio Grande do Sul,Rua 28 do Estado Rio Grande do Sul,805.0,Conjunto 13
12,2018-03-28,Carol,Barboza,carol@exemplo.com,9 8487-3501,Brasil,Mato Grosso,Rua 31 do Estado Mato Grosso,1.0,Conjunto 23


> Sempre é bom comprimir os dados pois os databricks cobra pelo amazenamento.

In [0]:
data.write.format("csv").options(compression="gzip").save("/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip")

In [0]:
display(dbutils.fs.ls("/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip"))

path,name,size,modificationTime
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/_SUCCESS,_SUCCESS,0,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/_committed_7736420262274967577,_committed_7736420262274967577,576,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/_started_7736420262274967577,_started_7736420262274967577,0,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/part-00000-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-204-1-c000.csv.gz,part-00000-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-204-1-c000.csv.gz,1059,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/part-00001-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-205-1-c000.csv.gz,part-00001-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-205-1-c000.csv.gz,811,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/part-00002-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-206-1-c000.csv.gz,part-00002-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-206-1-c000.csv.gz,774,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/part-00003-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-208-1-c000.csv.gz,part-00003-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-208-1-c000.csv.gz,602,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/part-00004-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-207-1-c000.csv.gz,part-00004-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-207-1-c000.csv.gz,516,1783638083000
dbfs:/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip/part-00005-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-209-1-c000.csv.gz,part-00005-tid-7736420262274967577-8ee39fb0-08a8-4045-ae86-518b61fc4660-209-1-c000.csv.gz,477,1783638083000


In [0]:
# Lendo arquivo .csv

data = spark\
        .read\
        .format("csv")\
        .options(
            header=True,
            inferSchema=False,
            delimiter=",",
            skipLeadingRows=1,
            compression="gzip"
        )\
        .schema(schema=schemas)\
        .load("/Volumes/learn_databricks/schema/volume/Clientes/partition_country_gzip")

Otimizar o armazenamento no Apache Spark usando compressão é um dos melhores caminhos para reduzir custos e, de bônus, ganhar muita performance (menos I/O de disco e rede).

Cada formato de arquivo (CSV, JSON, Parquet) se comporta de maneira diferente com os algoritmos de compressão. Vamos direto ao ponto: o que usar, quando usar e as vantagens de cada um.

---

## 1. Parquet (O Recomendado)

O Parquet é um formato colunar e já nasce com suporte nativo a compressões supereficientes.

* **Padrão do Spark:** **Snappy** (a partir do Spark 3.0, algumas distribuições usam Snappy ou Zstandard por padrão).
* **O que utilizar:** **Snappy** (para o dia a dia) ou **Zstandard (zstd)** (para dados frios/arquivamento).
* **Quando utilizar:** * **Snappy:** Quando você precisa de um ótimo equilíbrio entre velocidade de escrita/leitura e redução de tamanho. Ele é *splittable* (pode ser dividido em partes pelo Spark), o que permite processamento paralelo massivo.
* **Zstandard (zstd):** Quando o custo de armazenamento é crítico e você quer taxas de compressão altíssimas (próximas ao Gzip), mas sem perder tanta performance quanto perderia com o Gzip.



### Vantagens do Parquet + Snappy/Zstd

* **Alta Performance:** Como é colunar, o Spark só lê as colunas necessárias (*predicate pushdown*).
* **Divisível (*Splittable*):** O Spark consegue processar blocos do arquivo em paralelo, mesmo comprimido.

---

## 2. CSV e JSON (Formatos de Texto)

CSV e JSON são formatos baseados em linhas e texto puro. Eles não possuem compressão nativa por coluna, então comprimimos o arquivo inteiro.

* **O que utilizar:** **Gzip** ou **Bzip2** (se precisar dividir o arquivo).
* **Quando utilizar:**
* **Gzip:** Use quando os dados forem puramente para **armazenamento/backup (dados frios)** ou para exportação para sistemas externos.
* **Bzip2:** Use se você *precisa* processar arquivos CSV/JSON gigantes de forma paralela no Spark, pois o Bzip2 é um dos poucos que permite divisão (*splittability*) em formato texto.



### O grande problema do Gzip com CSV/JSON

> ⚠️ **Atenção:** Arquivos `.csv.gz` ou `.json.gz` **não são divisíveis (*non-splittable*)**. Se você tiver um arquivo Gzip de 10 GB, apenas **um único** Executor/Core do Spark vai processar esse arquivo inteiro, gerando um gargalo gigante (estouro de memória ou lentidão).

---

## Resumo Comparativo: Qual escolher?

| Formato de Arquivo | Algoritmo Recomendado | É Divisível (*Splittable*)? | Cenário de Uso |
| --- | --- | --- | --- |
| **Parquet** | **Snappy** | Sim | **Padrão ouro.** Use para quase tudo no ecossistema Spark (tabelas analíticas, queries frequentes). |
| **Parquet** | **Zstandard (zstd)** | Sim | Queries menos frequentes, dados históricos onde o espaço em disco é prioridade. |
| **CSV / JSON** | **Nenhum (Texto Limpo)** | Sim | Apenas se os arquivos forem pequenos e você precisar que qualquer ferramenta legível por humanos abra o arquivo. |
| **CSV / JSON** | **Gzip** | Não | Apenas para exportação de relatórios finais ou arquivamento (onde o Spark não vai reprocessar o arquivo constantemente). |

---

## Como aplicar no código (PySpark)

Para salvar seus dados aplicando a compressão correta:

```python
# Salvando em Parquet com Snappy (Geralmente já é o padrão)
df.write.mode("overwrite") \
        .option("compression", "snappy") \
        .parquet("/caminho/do/armazenamento/dados_parquet")

# Salvando em CSV comprimido com Gzip (Para exportação)
df.write.mode("overwrite") \
        .option("compression", "gzip") \
        .csv("/caminho/do/armazenamento/dados_csv")

```

**Dica de ouro:** Se você está sofrendo com tamanho de armazenamento, a melhor estratégia não é apenas mudar a compressão do CSV/JSON, mas sim **converter o seu pipeline para salvar em Parquet**. A diferença de tamanho e performance é brutal.